In [1]:
import glob
from operator import itemgetter

import geopandas as gpd
import numpy as np
import pandas as pd
from geopy.distance import geodesic

from core.abbreviations import clean_street_name

In [2]:
# Concat and get dataframe
df = pd.concat(
    [
        pd.read_csv(path, parse_dates=["date"])
        for path in glob.glob("../data/flat_price*_to_*.csv")
    ],
    ignore_index=True,
)

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Ave 1,10 To 12,31.0,Improved,1977,NaN,9000.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 977871 entries, 0 to 977870
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 977871 non-null  datetime64[us]
 1   town                 977871 non-null  str           
 2   flat_type            977871 non-null  str           
 3   block                977871 non-null  str           
 4   street_name          977871 non-null  str           
 5   storey_range         977871 non-null  str           
 6   floor_area_sqm       977871 non-null  float64       
 7   flat_model           977871 non-null  str           
 8   lease_commence_date  977871 non-null  int64         
 9   remaining_lease      268821 non-null  object        
 10  resale_price         977871 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(6)
memory usage: 128.5+ MB


In [4]:
# Check unique values for flat_type columns
df["flat_type"].unique()

<ArrowStringArray>
[          '1 Room',           '3 Room',           '4 Room',
           '5 Room',           '2 Room',        'Executive',
 'Multi Generation', 'Multi-Generation']
Length: 8, dtype: str

In [5]:
# Modify flat_type column by adding "-" for "Multi Generation"
mg_mask = pd.col("flat_type") == "Multi Generation"

df.loc[mg_mask, "flat_type"] = df.loc[mg_mask, "flat_type"].str.replace(" ", "-")

df["flat_type"].unique()

<ArrowStringArray>
[          '1 Room',           '3 Room',           '4 Room',
           '5 Room',           '2 Room',        'Executive',
 'Multi-Generation']
Length: 7, dtype: str

In [6]:
# Clean street name by expanding the abbreviations into expanding form
df = clean_street_name(df)

In [7]:
# Check unique values for flat_model columns
df["flat_model"].unique()

<ArrowStringArray>
[              'Improved',         'New Generation',                'Model A',
               'Standard',             'Simplified',     'Model A-Maisonette',
              'Apartment',             'Maisonette',                'Terrace',
                 '2-Room',    'Improved-Maisonette',       'Multi Generation',
      'Premium Apartment',          'Adjoined Flat',     'Premium Maisonette',
               'Model A2',                   'Dbss',                'Type S1',
                'Type S2', 'Premium Apartment Loft',                   '3Gen']
Length: 21, dtype: str

In [8]:
# Create price per sqm
df["price_per_sqm"] = df["resale_price"] / df["floor_area_sqm"]

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10 To 12,31.0,Improved,1977,NaN,9000.0,290.322581


In [9]:
# Create lower and upper bound columns for storey
df[["storey_lower", "storey_upper"]] = (
    df["storey_range"].str.split(" To ", expand=True).astype("int8")
)

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10 To 12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12


In [10]:
# Replace " TO " with  " - " for storey_range column
df["storey_range"] = df["storey_range"].str.replace(" To ", "-")

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12


In [11]:
# Crate year, quarter, and month
df["year"] = df["date"].dt.year
df["quarter"] = df["date"].dt.quarter
df["month"] = df["date"].dt.month

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper,year,quarter,month
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12,1990,1,1


In [12]:
# Split by " " and get the first index in the list
df["remaining_lease"] = df["remaining_lease"].str.split(" ").str[0]

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper,year,quarter,month
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12,1990,1,1


In [13]:
# Create full address column by concatenating block and street name
df["full_address"] = df["block"] + " " + df["street_name"]

df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper,year,quarter,month,full_address
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12,1990,1,1,309 Ang Mo Kio Avenue 1


## Get Price Index

In [14]:
# Load price index
pi_df = pd.read_csv("../data/resale_price_index.csv")

pi_df.head(1)

,year,quarter,index
0,1990,1,24.3


In [15]:
merged_df = df.merge(pi_df, how="left", on=["year", "quarter"])

merged_df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper,year,quarter,month,full_address,index
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12,1990,1,1,309 Ang Mo Kio Avenue 1,24.3


In [16]:
# Calculate adjusted price
merged_df["adjusted_price"] = np.where(
    merged_df["index"].isna(),
    merged_df["resale_price"],
    merged_df["resale_price"] * pi_df["index"].iloc[-1] / merged_df["index"],
)

merged_df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,price_per_sqm,storey_lower,storey_upper,year,quarter,month,full_address,index,adjusted_price
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,9000.0,290.322581,10,12,1990,1,1,309 Ang Mo Kio Avenue 1,24.3,75333.333333


## Get Planning Area and Region

In [17]:
# load planning area
pln_area_gdf = gpd.read_file("../data/pln_area.geojson")

pln_area_gdf.head(1)

,region,pln_area,geometry
0,Central,Bukit Timah,"POLYGON ((103.79644 1.34888, 103.79569 1.34917..."


In [18]:
# load address
address_df = gpd.read_file("../data/address.csv")

address_gdf = gpd.GeoDataFrame(
    address_df,
    geometry=gpd.points_from_xy(address_df["lng"], address_df["lat"]),
    crs="EPSG:4326",
)

address_gdf.head(1)

,full_address,lat,lng,geometry
0,309 Ang Mo Kio Avenue 1,1.365098,103.845337,POINT (103.84534 1.3651)


In [19]:
merged_gdf = gpd.sjoin(address_gdf, pln_area_gdf, how="left", predicate="within").drop(
    columns=["index_right"]
)

merged_gdf.head(1)

,full_address,lat,lng,geometry,region,pln_area
0,309 Ang Mo Kio Avenue 1,1.365098,103.845337,POINT (103.84534 1.3651),North-East,Ang Mo Kio


In [20]:
# load mrt
mrt_df = pd.read_csv("../data/mrt.csv")

mrt_df.head(1)

,name,lat,lng
0,Sungei Bedok,1.32139,103.958909


In [21]:
# Remove nan value in mrt_df
mrt_df = mrt_df.dropna(subset=["lat", "lng"])

mrt_df.head(1)

,name,lat,lng
0,Sungei Bedok,1.32139,103.958909


## Calculate the distance between Address and Mrt Station

In [22]:
RADIUS_KM = 2.0

results = []

for _, address in address_df.iterrows():
    address_coords = (address["lat"], address["lng"])

    nearest_station = None
    nearest_distance = np.inf
    stations_within = []  # collect all stations within 800m

    for _, station in mrt_df.iterrows():
        station_coords = (station["lat"], station["lng"])
        dist = geodesic(address_coords, station_coords).km

        # Track nearest station
        if dist < nearest_distance:
            nearest_distance = dist
            nearest_station = station["name"]

        # Collect all within 800m
        if dist <= RADIUS_KM:  # check every station, not just nearest
            stations_within.append({"name": station["name"], "dist": round(dist, 4)})

    # Sort within-800m list by distance, nearest first
    stations_within = sorted(stations_within, key=itemgetter("dist"))

    results.append(
        {
            "nearest_station": nearest_station
            if nearest_distance <= RADIUS_KM
            else np.nan,
            "radius_km": round(nearest_distance, 4)
            if nearest_distance <= RADIUS_KM
            else np.nan,
            "stations_within_2km": [station["name"] for station in stations_within],
            "station_count": len(stations_within),
        }
    )


In [23]:
final_gdf = pd.concat([merged_gdf, pd.DataFrame(results)], axis=1)

final_gdf.head(1)

,full_address,lat,lng,geometry,region,pln_area,nearest_station,radius_km,stations_within_2km,station_count
0,309 Ang Mo Kio Avenue 1,1.365098,103.845337,POINT (103.84534 1.3651),North-East,Ang Mo Kio,Teck Ghee,0.2652,"[Teck Ghee, Ang Mo Kio, Mayflower, Bright Hill...",8


In [24]:
# Merge main dataframe with final_gdf
final_df = merged_df.merge(final_gdf, how="left", on="full_address")

final_df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,adjusted_price,lat,lng,geometry,region,pln_area,nearest_station,radius_km,stations_within_2km,station_count
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,...,75333.333333,1.365098,103.845337,POINT (103.84534 1.3651),North-East,Ang Mo Kio,Teck Ghee,0.2652,"[Teck Ghee, Ang Mo Kio, Mayflower, Bright Hill...",8


In [25]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 977871 entries, 0 to 977870
Data columns (total 29 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 977871 non-null  datetime64[us]
 1   town                 977871 non-null  str           
 2   flat_type            977871 non-null  str           
 3   block                977871 non-null  str           
 4   street_name          977871 non-null  str           
 5   storey_range         977871 non-null  str           
 6   floor_area_sqm       977871 non-null  float64       
 7   flat_model           977871 non-null  str           
 8   lease_commence_date  977871 non-null  int64         
 9   remaining_lease      231668 non-null  object        
 10  resale_price         977871 non-null  float64       
 11  price_per_sqm        977871 non-null  float64       
 12  storey_lower         977871 non-null  int8          
 13  storey_upper         9778

In [26]:
# Remove columns
final_df = final_df.loc[
    :,
    ~final_df.columns.str.contains(
        r"^lat$|^lng$|geometry|index|stations_within_2km", regex=True
    ),
]

final_df.head(1)

,date,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,year,quarter,month,full_address,adjusted_price,region,pln_area,nearest_station,radius_km,station_count
0,1990-01-01,Ang Mo Kio,1 Room,309,Ang Mo Kio Avenue 1,10-12,31.0,Improved,1977,NaN,...,1990,1,1,309 Ang Mo Kio Avenue 1,75333.333333,North-East,Ang Mo Kio,Teck Ghee,0.2652,8


In [27]:
# Check whether there is a duplication
final_df.duplicated().sum()

np.int64(2003)

In [28]:
# Remove duplication
final_df = final_df.drop_duplicates()

final_df.duplicated().sum()

np.int64(0)

In [36]:
# Save final_df to csv
final_df.to_csv("../data/latest_data.csv", index=False)

In [ ]:
final_df.to_parquet("../data/latest_data.parquet", engine="pyarrow", index=False)
